# MA-EZV2 Demo

This notebook demonstrates instantiation of the MA-EZV2 policy, running MCTS, and plotting simple diagnostics. It is a non-executed, ready-to-run notebook.


In [ ]:
import yaml
import os
import torch
import matplotlib.pyplot as plt

from paperAssignments.Assignments1_50.CA10.integration.lightzero_adapter import (
    LightZeroAdapter,
)

cfg_path = os.path.join(
    "..",
    "paperAssignments",
    "Assignments1-50",
    "CA10",
    "configs",
    "ma_ezv2_default.yaml",
)
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

adapter = LightZeroAdapter(cfg, device="cpu")
obs = torch.randn(1, cfg["model"]["obs_dim"])
info = adapter.infer(obs)
print("root value:", info["value"])

In [ ]:
res = adapter.search(obs, sims=30, topk=6)
# adapter.search now returns (visits, policy, joint_visits) optionally
+    if isinstance(res, tuple) and len(res) == 3:
+    
+        visits = res[0].squeeze(0).numpy()
+        policy = res[1].squeeze(0).numpy()
+        joint = res[2][0]  # (keys, vals)
+    else:
+        visits = res["visits"].squeeze(0).numpy()
+        policy = res["policy"].squeeze(0).numpy()
+        joint = None

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(visits)), visits)
plt.title("Visit counts")
plt.subplot(1, 2, 2)
plt.bar(range(len(policy)), policy)
plt.title("Policy from visits")
plt.tight_layout()
plt.savefig("../pictures/ma_ezv2_demo.png", dpi=200)
print("Saved figure to ../pictures/ma_ezv2_demo.png")
if joint is not None:
    keys, vals = joint
    if len(keys) > 0:
        # plot joint visit counts as bar chart (sparse)
        plt.figure(figsize=(8,4))
        labels = [str(k) for k in keys]
        plt.bar(range(len(vals)), vals.numpy())
        plt.xticks(range(len(vals)), labels, rotation=90)
        plt.tight_layout()
        plt.savefig('../pictures/ma_ezv2_joint_visits.png', dpi=200)
        print('Saved joint visits to ../pictures/ma_ezv2_joint_visits.png')

## Training step example

Below is a template cell showing how to call adapter.training_step in a training loop. It's not executed here.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch

ds = TensorDataset(
    torch.randn(32, cfg["model"]["obs_dim"]),
    torch.randn(32, cfg["model"]["joint_action_dim"]),
    torch.softmax(torch.randn(32, cfg["model"]["joint_action_dim"]), dim=-1),
    torch.randn(32),
    torch.randn(32),
    torch.randn(32),
)
loader = DataLoader(ds, batch_size=8)
optim = torch.optim.Adam(adapter.policy.parameters(), lr=1e-3)
loss_weights = cfg.get("loss_weights", {})
for batch in loader:
    obs_b, actions_b, pi_b, v_b, r_b, z_b = [b for b in batch]
    batch_dict = {
        "obs": obs_b,
        "actions": actions_b,
        "pi_target": pi_b,
        "v_target": v_b,
        "r_target": r_b,
        "z_target": z_b,
    }
    loss_val, loss_terms = adapter.training_step(batch_dict, loss_weights, optim)
    print("loss", loss_val)

### Synthetic Training Visualization

This cell simulates training metrics (losses, win rate) and plots them for demonstration. Replace synthetic data with real logged metrics when running experiments.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Synthetic metrics (replace with real logging in practice)
steps = np.arange(0, 500)
loss_total = 1.0 / (1.0 + 0.01 * steps) + 0.02 * np.random.randn(len(steps))
loss_pi = 0.5 / (1.0 + 0.01 * steps) + 0.01 * np.random.randn(len(steps))
loss_v = 0.3 / (1.0 + 0.008 * steps) + 0.01 * np.random.randn(len(steps))
win_rate = np.clip(0.2 + 0.001 * steps + 0.02 * np.random.randn(len(steps)), 0, 1)

plt.figure(figsize=(10, 4))
plt.plot(steps, loss_total, label="loss_total")
plt.plot(steps, loss_pi, label="loss_pi")
plt.plot(steps, loss_v, label="loss_v")
plt.xlabel("training steps")
plt.ylabel("loss")
plt.legend()
plt.title("Training Loss Curves (synthetic)")
plt.savefig("../pictures/ma_ezv2_loss_curves.png", dpi=200)
print("Saved loss curves to ../pictures/ma_ezv2_loss_curves.png")

plt.figure(figsize=(6, 3))
plt.plot(steps, win_rate)
plt.xlabel("training steps")
plt.ylabel("win rate")
plt.title("Win rate over training (synthetic)")
plt.savefig("../pictures/ma_ezv2_win_rate.png", dpi=200)
print("Saved win rate plot to ../pictures/ma_ezv2_win_rate.png")

### MCTS Visit Heatmap and Beam Frequency

Visualize visit distributions across moves and beam candidate frequencies.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Synthetic visit matrix: moves x actions
moves = 50
actions = cfg["model"]["joint_action_dim"]
visits_matrix = np.abs(np.random.randn(moves, actions))

plt.figure(figsize=(8, 4))
plt.imshow(visits_matrix.T, aspect="auto", cmap="viridis")
plt.colorbar(label="visits")
plt.xlabel("move")
plt.ylabel("action")
plt.title("MCTS visit heatmap (synthetic)")
plt.tight_layout()
plt.savefig("../pictures/ma_ezv2_visit_heatmap.png", dpi=200)
print("Saved visit heatmap to ../pictures/ma_ezv2_visit_heatmap.png")

# Beam frequency
beam_candidates = ["a" + str(i) for i in range(actions)]
freqs = np.random.randint(0, 50, size=len(beam_candidates))
plt.figure(figsize=(8, 3))
plt.bar(range(len(freqs)), freqs)
plt.xticks(range(len(freqs)), beam_candidates, rotation=90)
plt.title("Beam candidate frequencies (synthetic)")
plt.tight_layout()
plt.savefig("../pictures/ma_ezv2_beam_freqs.png", dpi=200)
print("Saved beam frequencies to ../pictures/ma_ezv2_beam_freqs.png")